In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'SAE.py').exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
MNIST_PIPELINE_DIR = REPO_ROOT / 'mnist' / 'pipeline'
if str(MNIST_PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(MNIST_PIPELINE_DIR))

import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import torch
try:
    from IPython.display import display
except ImportError:
    display = print
from scipy.spatial.distance import cdist
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

import ot

try:
    import umap
except ImportError:
    umap = None

from parameter_search import load_with_labels, make_stratified_split
from mnist_sae_models import DisplacementFieldSAE


def auto_lims(*point_arrays, pad_frac=0.1):
    all_pts = np.concatenate(point_arrays, axis=0)
    xmin, ymin = all_pts.min(axis=0)
    xmax, ymax = all_pts.max(axis=0)
    dx = (xmax - xmin) * pad_frac / 2
    dy = (ymax - ymin) * pad_frac / 2
    cx, cy = (xmin + xmax) / 2, (ymin + ymax) / 2
    half = max(xmax - xmin, ymax - ymin) / 2 + max(dx, dy)
    return cx - half, cx + half, cy - half, cy + half


def set_lims(ax, lims):
    ax.set_xlim(lims[0], lims[1])
    ax.set_ylim(lims[2], lims[3])
    ax.set_aspect("equal")


def load_trial_cfg(search_dir, trial_name):
    trials_path = search_dir / "trials.json"
    if trials_path.exists():
        with open(trials_path, "r") as f:
            trials = json.load(f)
        for trial in trials:
            if trial["name"] == trial_name:
                return trial
        raise KeyError(f"Trial {trial_name!r} not found in {trials_path}")

    # Fallback: train_mnist_sae.py format (config.json + optional metrics.json)
    config_path = search_dir / "config.json"
    cfg = json.loads(config_path.read_text()) if config_path.exists() else {}

    run_cfg = {}
    metrics_path = search_dir / "metrics.json"
    if metrics_path.exists():
        with open(metrics_path, "r") as f:
            metrics = json.load(f)
        for run in metrics:
            rname = f"{run['method']}_eps{run['epsilon']}_c{run['sparsity_coeff']}"
            if rname == trial_name:
                run_cfg = run
                break

    eps_default = cfg.get("epsilons", [0.025])[0]
    c_default = cfg.get("sparsity_coeffs", [0.001])[0]
    return {
        "name": trial_name,
        "m": cfg.get("m", 30),
        "eps": run_cfg.get("epsilon", eps_default),
        "grid_side": cfg.get("grid_side", 64),
        "lista_steps": cfg.get("lista_steps", 20),
        "activation_type": cfg.get("activation_type", "relu"),
        "normalize_atoms": cfg.get("normalize_atoms", True),
        "atoms_type": cfg.get("atoms_type", "gibbs"),
        "n_sinkhorn": cfg.get("n_sinkhorn", 30),
        "topk_k": cfg.get("topk_k", 3),
        "per_atom_gain": cfg.get("per_atom_gain", True),
        "lateral_init": cfg.get("lateral_init", "damped_identity"),
        "sparsity_coeff": run_cfg.get("sparsity_coeff", c_default),
        "batch_size": cfg.get("batch_size", 128),
        "data_dir": cfg.get("data_dir"),
    }


def repo_path(path):
    """Resolve relative paths from the repo root, not the notebook launch directory."""
    path = Path(path)
    return path if path.is_absolute() else REPO_ROOT / path


def resolve_data_dir(trial_cfg, default_data_dir):
    """Return the trial data_dir, falling back to the notebook default."""
    raw = trial_cfg.get("data_dir")
    if raw is None:
        return repo_path(default_data_dir)
    return repo_path(raw)


def locate_checkpoint(search_dir, trial_name):
    trial_dir = search_dir / trial_name
    candidates = [
        trial_dir / "model_latest.pt",
        trial_dir / "latest_state.pt",
    ]
    for path in candidates:
        if path.exists():
            return path

    staged = sorted(trial_dir.glob("model_stage*.pt"))
    if staged:
        return staged[-1]

    epoch_ckpts = sorted(trial_dir.glob("model_epoch*.pt"))
    if epoch_ckpts:
        return epoch_ckpts[-1]

    # Fallback: train_mnist_sae.py saves flat {trial_name}.pt in the search_dir
    flat_ckpt = search_dir / f"{trial_name}.pt"
    if flat_ckpt.exists():
        return flat_ckpt

    raise FileNotFoundError(
        f"Could not find a checkpoint for {trial_name} under {trial_dir} "
        f"or as {flat_ckpt}. "
        "Expected model_latest.pt, latest_state.pt, model_stage*.pt, model_epoch*.pt, "
        "or a flat {trial_name}.pt in the search directory."
    )


def load_state_dict(checkpoint_path):
    payload = torch.load(checkpoint_path, map_location="cpu")
    if isinstance(payload, dict) and "model_state" in payload:
        return payload["model_state"], payload
    return payload, payload


def build_model_from_trial(cfg, X, device):
    model = DisplacementFieldSAE(
        X,
        m=cfg["m"],
        eps=cfg["eps"],
        grid_side=cfg["grid_side"],
        lista_steps=cfg["lista_steps"],
        activation_type=cfg.get("activation_type", "relu"),
        normalize_atoms=cfg.get("normalize_atoms", False),
        grid_points=None,
        atoms_type=cfg.get("atoms_type", "gibbs"),
        n_sinkhorn=cfg.get("n_sinkhorn", 30),
        topk_k=cfg.get("topk_k", 3),
        per_atom_gain=cfg.get("per_atom_gain", False),
        lateral_init=cfg.get("lateral_init", "zeros"),
    ).to(device)
    return model


def build_model_from_checkpoint(cfg, state_dict, X, device):
    """Build model + load weights, inferring all structural params from the state dict.

    Returns (model, load_result, updated_cfg) — updated_cfg has the inferred values
    so downstream summary prints stay accurate even with no config.json.
    """
    if "atoms_module.H_raw" in state_dict:
        atoms_type = "gibbs"
        m = state_dict["atoms_module.H_raw"].shape[0]
    elif "atoms_module.psi" in state_dict:
        atoms_type = "sinkhorn"
        m = state_dict["atoms_module.psi"].shape[0]
    else:
        atoms_type = cfg.get("atoms_type", "gibbs")
        m = cfg.get("m", 30)

    K = state_dict["atoms_module.Y"].shape[0]
    grid_side = int(K ** 0.5)

    lista_steps = sum(1 for k in state_dict if k.startswith("encoder.biases.")) or cfg.get("lista_steps", 20)
    per_atom_gain = "encoder.gain" in state_dict
    activation_type = "jumprelu" if any(k.startswith("encoder.thresholds.") for k in state_dict) else cfg.get("activation_type", "relu")

    inferred = dict(cfg, m=m, grid_side=grid_side, lista_steps=lista_steps,
                    per_atom_gain=per_atom_gain, atoms_type=atoms_type,
                    activation_type=activation_type)

    model = DisplacementFieldSAE(
        X,
        m=inferred["m"],
        eps=inferred.get("eps", 0.025),
        grid_side=inferred["grid_side"],
        lista_steps=inferred["lista_steps"],
        activation_type=inferred["activation_type"],
        normalize_atoms=inferred.get("normalize_atoms", True),
        grid_points=None,
        atoms_type=inferred["atoms_type"],
        n_sinkhorn=inferred.get("n_sinkhorn", 30),
        topk_k=inferred.get("topk_k", 3),
        per_atom_gain=inferred["per_atom_gain"],
        lateral_init=inferred.get("lateral_init", "damped_identity"),
    ).to(device)

    load_result = model.load_state_dict(state_dict, strict=False)
    return model, load_result, inferred


@torch.no_grad()
def forward_dataset(model, maps, device, batch_size=128):
    model.eval()
    n = model.n
    all_recons, all_codes, all_recon_loss = [], [], []
    for start in range(0, maps.shape[0], batch_size):
        batch = maps[start:start + batch_size].to(device)
        recon, codes = model(batch)
        per_sample_recon = 0.5 * ((batch - recon) ** 2).sum(dim=(1, 2)) / n
        all_recons.append(recon.cpu())
        all_codes.append(codes.cpu())
        all_recon_loss.append(per_sample_recon.cpu())
    return (
        torch.cat(all_recons, dim=0),
        torch.cat(all_codes, dim=0),
        torch.cat(all_recon_loss, dim=0),
    )


def pick_indices_per_class(labels, n_per_class=1, digits=None, seed=0):
    rng = np.random.RandomState(seed)
    labels_np = labels.cpu().numpy() if isinstance(labels, torch.Tensor) else np.asarray(labels)
    classes = sorted(np.unique(labels_np).tolist())
    if digits is not None:
        classes = [d for d in digits if d in classes]

    picked = []
    for digit in classes:
        digit_idx = np.where(labels_np == digit)[0]
        take = min(n_per_class, len(digit_idx))
        if take == 0:
            continue
        chosen = rng.choice(digit_idx, size=take, replace=False)
        picked.extend(sorted(chosen.tolist()))
    return picked


def top_active_atoms(code_vec, top_k=5):
    code_np = np.asarray(code_vec)
    active = np.where(code_np > 0)[0]
    if len(active) == 0:
        return "none"
    ranked = sorted(active, key=lambda j: code_np[j], reverse=True)[:top_k]
    return ", ".join(f"#{j}:{code_np[j]:.3f}" for j in ranked)


def empirical_wasserstein_distance(orig, recon):
    n = orig.shape[0]
    a = np.full(n, 1.0 / n)
    b = np.full(n, 1.0 / n)
    cost = cdist(orig, recon, metric="sqeuclidean")
    return float(ot.emd2(a, b, cost))


def pointwise_l2_distance(orig, recon):
    """
    orig[i] and recon[i] are both the images of the same base point X[i].
    Returns the RMS pointwise error across base-point indices.
    """
    diff = orig - recon                  # shape (n, 2)
    sq_norms = np.sum(diff**2, axis=1)   # shape (n,)
    return float(np.sqrt(np.mean(sq_norms)))

def make_embedding_df(embedding, labels, prefix):
    data = {f"{prefix}{i + 1}": embedding[:, i] for i in range(embedding.shape[1])}
    data["digit"] = labels.astype(str)
    data["sample_index"] = np.arange(len(labels))
    return pd.DataFrame(data)


def maybe_subsample(codes, labels, max_points=None, seed=0):
    if max_points is None or max_points >= len(labels):
        return codes, labels

    rng = np.random.RandomState(seed)
    labels_np = labels.cpu().numpy() if isinstance(labels, torch.Tensor) else np.asarray(labels)
    classes = sorted(np.unique(labels_np).tolist())
    per_class = max(1, max_points // max(len(classes), 1))
    keep = []
    for digit in classes:
        idx = np.where(labels_np == digit)[0]
        take = min(per_class, len(idx))
        keep.extend(rng.choice(idx, size=take, replace=False).tolist())
    keep = np.array(sorted(keep))
    return codes[keep], labels[keep]


def plot_base_measure(X, title_prefix="", alpha=0.8, figsize=(4, 4)):
    """Scatter the base-measure support points X (n, 2)."""
    pts = X.detach().cpu().numpy() if isinstance(X, torch.Tensor) else np.asarray(X)
    lims = auto_lims(pts)
    fig, ax = plt.subplots(figsize=figsize)
    ax.scatter(
            pts[:, 0],
            pts[:, 1],
            s=5,
            c=pts[:, 0],
            cmap="viridis",
            alpha=alpha,
        )
    title = "Base measure" if not title_prefix else f"{title_prefix} | base measure"
    ax.set_title(f"{title}  (n={pts.shape[0]})")
    set_lims(ax, lims)
    ax.set_xticks([])
    ax.set_yticks([])
    plt.tight_layout()
    plt.show()


def plot_atoms(model, cfg, alpha=0.8, cols=5):
    with torch.no_grad():
        atoms = model.atoms_module().detach().cpu().numpy()
    base = model.atoms_module.X.detach().cpu().numpy()
    m = atoms.shape[0]

    all_atom_pts = atoms.reshape(-1, 2)
    lims = auto_lims(all_atom_pts)

    rows = int(np.ceil(m / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows), squeeze=False)
    axes = axes.flatten()
    for j in range(m):
        ax = axes[j]
        ax.scatter(
            atoms[j, :, 0],
            atoms[j, :, 1],
            s=5,
            c=base[:, 0],
            cmap="viridis",
            alpha=alpha,
        )
        set_lims(ax, lims)
        ax.set_xticks([])
        ax.set_yticks([])
    for j in range(m, len(axes)):
        axes[j].axis("off")
    plt.tight_layout()
    plt.show()


def plot_reconstructions(maps, labels, recons, codes, n_per_class=1, alpha=0.6, digits=None, seed=0):
    indices = pick_indices_per_class(labels, n_per_class=n_per_class, digits=digits, seed=seed)
    if not indices:
        raise ValueError("No reconstruction examples were selected.")

    maps_np = maps[indices].cpu().numpy()
    recons_np = recons[indices].cpu().numpy()
    labels_np = labels[indices].cpu().numpy()
    codes_np = codes[indices].cpu().numpy()

    fig, axes = plt.subplots(len(indices), 2, figsize=(10, 4 * len(indices)), squeeze=False)
    for row, (orig, recon, digit, code) in enumerate(zip(maps_np, recons_np, labels_np, codes_np)):
        lims = auto_lims(orig, recon)
        w2 = empirical_wasserstein_distance(orig, recon)**.5
        l2_map = pointwise_l2_distance(orig, recon)

        ax_left = axes[row, 0]
        ax_left.scatter(orig[:, 0], orig[:, 1], s=4, alpha=alpha, color="tab:blue")
        ax_left.set_title(f"Digit {digit} | original")
        set_lims(ax_left, lims)
        ax_left.set_xticks([])
        ax_left.set_yticks([])

        ax_right = axes[row, 1]
        ax_right.scatter(recon[:, 0], recon[:, 1], s=4, alpha=alpha, color="tab:orange")
        ax_right.set_title(f"Digit {digit} | reconstructed  (W2={w2:.4f}, L2={l2_map:.4f})")
        set_lims(ax_right, lims)
        ax_right.set_xticks([])
        ax_right.set_yticks([])

    plt.tight_layout()
    plt.show()


def plot_code_heatmap(codes, labels, max_samples=300, seed=0):
    sub_codes, sub_labels = maybe_subsample(codes, labels, max_points=max_samples, seed=seed)
    codes_np = sub_codes.cpu().numpy()
    labels_np = sub_labels.cpu().numpy()
    order = np.lexsort((np.arange(len(labels_np)), labels_np))
    codes_np = codes_np[order]
    labels_np = labels_np[order]

    fig, ax = plt.subplots(figsize=(12, 6))
    im = ax.imshow(codes_np, aspect="auto", interpolation="nearest", cmap="magma")
    ax.set_title("Code heatmap (samples sorted by digit)")
    ax.set_xlabel("Atom index")
    ax.set_ylabel("Sample index")

    boundaries = np.where(np.diff(labels_np) != 0)[0]
    for b in boundaries:
        ax.axhline(b + 0.5, color="white", linewidth=0.8, alpha=0.8)

    unique_digits = np.unique(labels_np)
    tick_positions = []
    tick_labels = []
    for digit in unique_digits:
        idx = np.where(labels_np == digit)[0]
        tick_positions.append(idx.mean())
        tick_labels.append(str(digit))
    ax.set_yticks(tick_positions)
    ax.set_yticklabels(tick_labels)

    cbar = fig.colorbar(im, ax=ax, fraction=0.02, pad=0.02)
    cbar.set_label("Code magnitude")
    plt.tight_layout()
    plt.show()


def plot_code_pca(codes, labels, title_prefix, max_points=None, seed=0):
    sub_codes, sub_labels = maybe_subsample(codes, labels, max_points=max_points, seed=seed)
    pca = PCA(n_components=3)
    pcs = pca.fit_transform(sub_codes.cpu().numpy())
    df = make_embedding_df(pcs, sub_labels.cpu().numpy(), prefix="PC")

    title_2d = (
        f"{title_prefix} | 2D PCA "
        f"(var={pca.explained_variance_ratio_[0]:.2%}, {pca.explained_variance_ratio_[1]:.2%})"
    )
    fig2 = px.scatter(
        df,
        x="PC1",
        y="PC2",
        color="digit",
        title=title_2d,
        hover_data=["sample_index"],
        opacity=0.75,
    )
    fig2.update_traces(marker=dict(size=6))
    fig2.show()

    title_3d = (
        f"{title_prefix} | 3D PCA "
        f"(var={pca.explained_variance_ratio_[0]:.2%}, {pca.explained_variance_ratio_[1]:.2%}, {pca.explained_variance_ratio_[2]:.2%})"
    )
    fig3 = px.scatter_3d(
        df,
        x="PC1",
        y="PC2",
        z="PC3",
        color="digit",
        title=title_3d,
        hover_data=["sample_index"],
        opacity=0.7,
    )
    fig3.update_traces(marker=dict(size=3))
    fig3.show()


def plot_code_umap(codes, labels, title_prefix, max_points=None, seed=0, n_neighbors=15, min_dist=0.1):
    if umap is None:
        print("UMAP skipped: install `umap-learn` in this environment to enable this plot.")
        return

    sub_codes, sub_labels = maybe_subsample(codes, labels, max_points=max_points, seed=seed)
    reducer = umap.UMAP(n_components=2, random_state=seed, n_neighbors=n_neighbors, min_dist=min_dist)
    embedding = reducer.fit_transform(sub_codes.cpu().numpy())
    df = make_embedding_df(embedding, sub_labels.cpu().numpy(), prefix="UMAP")

    fig = px.scatter(
        df,
        x="UMAP1",
        y="UMAP2",
        color="digit",
        title=f"{title_prefix} | UMAP",
        hover_data=["sample_index"],
        opacity=0.75,
    )
    fig.update_traces(marker=dict(size=6))
    fig.show()


def fit_logistic_and_plot(train_codes, train_labels, eval_codes, eval_labels, title, max_iter=4000):
    clf = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=max_iter),
    )
    clf.fit(train_codes, train_labels)
    preds = clf.predict(eval_codes)
    acc = accuracy_score(eval_labels, preds)
    print(f"{title} accuracy: {acc:.4f}")
    print(classification_report(eval_labels, preds, digits=4))

    fig, ax = plt.subplots(figsize=(6, 6))
    ConfusionMatrixDisplay.from_predictions(
        eval_labels,
        preds,
        ax=ax,
        cmap="Blues",
        colorbar=False,
    )
    ax.set_title(title)
    plt.tight_layout()
    plt.show()
    return clf, preds, acc


def fit_logistic_random_split_and_plot(codes, labels, title, test_size=0.2, seed=0, max_iter=4000):
    codes_np = codes.cpu().numpy() if isinstance(codes, torch.Tensor) else np.asarray(codes)
    labels_np = labels.cpu().numpy() if isinstance(labels, torch.Tensor) else np.asarray(labels)
    train_codes, test_codes, train_labels, test_labels = train_test_split(
        codes_np,
        labels_np,
        test_size=test_size,
        random_state=seed,
        stratify=labels_np,
    )
    return fit_logistic_and_plot(
        train_codes=train_codes,
        train_labels=train_labels,
        eval_codes=test_codes,
        eval_labels=test_labels,
        title=title,
        max_iter=max_iter,
    )


In [ ]:
# Two trial sources are supported:
#   - SEARCH_DIR points to a train_mnist_sae.py output dir  (config.json + flat {run_name}.pt files)
#   - SEARCH_DIR can also point to a legacy lista_search run  (many lista_search_### trials)
#     Trial name format: "{method}_eps{eps}_c{sparsity_coeff}"
#     e.g. "displacement_eps0.025_c0.001"

SEARCH_DIR = repo_path("datasets/mnist_ot/SAE_params")

# Default dataset for trials that do not persist their own data_dir.
DATA_DIR_DEFAULT = repo_path("datasets/mnist_ot")

# Prefer CPU for notebook visualization on Apple Silicon; the MNIST SAE model has hit MPS compiler failures.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Set to a direct .pt path to load a train_mnist_sae.py checkpoint.
# When set, SEARCH_DIR and SELECTED_TRIAL are ignored.
# e.g. Path("datasets/mnist_ot/SAE_params/displacement_eps0.025_c0.0001.pt")
DIRECT_CHECKPOINT = None

CANDIDATE_TRIALS = [
    {"name": "displacement_eps0.025_c0.0001", "test_recon": 0.000568, "eps": 0.025, "c": 0.0001},
]

SELECTED_TRIAL = (
    json.loads((SEARCH_DIR / "best_trial.json").read_text())["name"]
    if (SEARCH_DIR / "best_trial.json").exists()
    else "displacement_eps0.025_c0.0001"
)

ATOM_ALPHA = 0.8
RECON_ALPHA = 0.55
RECONS_PER_DIGIT = 1
RECON_DIGITS = None      # e.g. [0, 1, 2, 3]
RECON_SEED = 0

HEATMAP_MAX_SAMPLES = 300
HEATMAP_SEED = 0

PCA_MAX_POINTS = None    # e.g. 2500 if you want a lighter Plotly figure
PCA_SEED = 0

UMAP_MAX_POINTS = 3000
UMAP_SEED = 0
UMAP_N_NEIGHBORS = 15
UMAP_MIN_DIST = 0.1

LOGREG_TEST_SIZE = 0.2
LOGREG_SEED = 0
LOGREG_MAX_ITER = 4000

SEARCH_TEST_FRACTION = None   # set, for example, to 0.1 if you know the original search value
SEARCH_SPLIT_SEED = None      # set, for example, to 42 if you know the original search seed

display(pd.DataFrame(CANDIDATE_TRIALS))
print(f"Using device: {DEVICE}")
print(f"SEARCH_DIR:  {SEARCH_DIR}")
print(f"Selected trial: {SELECTED_TRIAL}")
print(f"UMAP available: {umap is not None}")


In [ ]:
if DIRECT_CHECKPOINT is not None:
    _ckpt = Path(DIRECT_CHECKPOINT)
    trial_cfg = load_trial_cfg(_ckpt.parent, _ckpt.stem)
    checkpoint_path = _ckpt
else:
    trial_cfg = load_trial_cfg(SEARCH_DIR, SELECTED_TRIAL)
    checkpoint_path = locate_checkpoint(SEARCH_DIR, SELECTED_TRIAL)

state_dict, _ = load_state_dict(checkpoint_path)
data_dir = resolve_data_dir(trial_cfg, DATA_DIR_DEFAULT)
print(f"Loading data from: {data_dir}")

X, maps, labels = load_with_labels(data_dir)
model, load_result, trial_cfg = build_model_from_checkpoint(trial_cfg, state_dict, X, DEVICE)
model.eval()

T_hat_all, codes_all, recon_loss_all = forward_dataset(
    model,
    maps,
    DEVICE,
    batch_size=trial_cfg.get("batch_size", 128),
)

print(json.dumps({
    "checkpoint_path": str(checkpoint_path),
    "data_dir": str(data_dir),
    "eps": trial_cfg["eps"],
    "sparsity_coeff": trial_cfg["sparsity_coeff"],
    "lista_steps": trial_cfg["lista_steps"],
    "m": trial_cfg.get("m"),
    "mean_recon_all_data": float(recon_loss_all.mean()),
    "mean_active_atoms_all_data": float((codes_all > 0).float().sum(dim=1).mean()),
}, indent=2))
print(load_result)


In [ ]:
# --- Average pointwise L2 error across the full dataset ---
maps_np_all = maps.cpu().numpy()
recons_np_all = T_hat_all.cpu().numpy()

per_sample_l2 = np.array([
    pointwise_l2_distance(maps_np_all[i], recons_np_all[i])
    for i in range(len(maps_np_all))
])

labels_np_all = labels.cpu().numpy() if isinstance(labels, torch.Tensor) else np.asarray(labels)

print(f"Mean pointwise L2 error (all data): {per_sample_l2.mean():.6f}")
print(f"Std  pointwise L2 error (all data): {per_sample_l2.std():.6f}")
print()
print("Per-digit mean L2:")
for digit in sorted(np.unique(labels_np_all)):
    mask = labels_np_all == digit
    print(f"  Digit {digit}: {per_sample_l2[mask].mean():.6f}")


In [ ]:
plot_base_measure(X, title_prefix=SELECTED_TRIAL)


In [ ]:
plot_atoms(model, trial_cfg, alpha=ATOM_ALPHA, cols=5)


In [ ]:
plot_reconstructions(
    maps,
    labels,
    T_hat_all,
    codes_all,
    n_per_class=RECONS_PER_DIGIT,
    alpha=RECON_ALPHA,
    digits=RECON_DIGITS,
    seed=RECON_SEED,
)


In [ ]:
plot_code_heatmap(
    codes_all,
    labels,
    max_samples=HEATMAP_MAX_SAMPLES,
    seed=HEATMAP_SEED,
)


In [ ]:
plot_code_pca(
    codes_all,
    labels,
    title_prefix=f"{SELECTED_TRIAL} codes",
    max_points=PCA_MAX_POINTS,
    seed=PCA_SEED,
)

plot_code_umap(
    codes_all,
    labels,
    title_prefix=f"{SELECTED_TRIAL} codes",
    max_points=UMAP_MAX_POINTS,
    seed=UMAP_SEED,
    n_neighbors=UMAP_N_NEIGHBORS,
    min_dist=UMAP_MIN_DIST,
)


In [ ]:
_ = fit_logistic_random_split_and_plot(
    codes=codes_all,
    labels=labels,
    title=f"{SELECTED_TRIAL} logistic classifier on an 80/20 random split",
    test_size=LOGREG_TEST_SIZE,
    seed=LOGREG_SEED,
    max_iter=LOGREG_MAX_ITER,
)

if SEARCH_TEST_FRACTION is not None and SEARCH_SPLIT_SEED is not None:
    split = make_stratified_split(
        maps,
        labels,
        test_fraction=SEARCH_TEST_FRACTION,
        seed=SEARCH_SPLIT_SEED,
    )
    _, train_codes, _ = forward_dataset(model, split["train_maps"], DEVICE, batch_size=trial_cfg.get("batch_size", 128))
    _, test_codes, _ = forward_dataset(model, split["test_maps"], DEVICE, batch_size=trial_cfg.get("batch_size", 128))

    _ = fit_logistic_and_plot(
        train_codes=train_codes.cpu().numpy(),
        train_labels=split["train_labels"].cpu().numpy(),
        eval_codes=test_codes.cpu().numpy(),
        eval_labels=split["test_labels"].cpu().numpy(),
        title=f"{SELECTED_TRIAL} logistic classifier on reconstructed lista_search holdout",
        max_iter=LOGREG_MAX_ITER,
    )
else:
    print(
        "Skipping reconstructed lista_search holdout evaluation. The original split is not persisted in search_dir; "
        "set SEARCH_TEST_FRACTION and SEARCH_SPLIT_SEED above if you know them and want to reconstruct it."
    )


In [ ]:
# --- kNN density tuning ---
# Pick a single reconstructed sample and plot a histogram of its point densities
# to help choose a good k and bottom-percentile threshold.
#
# Density estimator:  rho_i ∝ (1 / mean_{j in kNN(i)} ||x_i - x_j||)^d

from scipy.spatial import KDTree

KNN_DIGIT      = 2    # which digit class to sample from
KNN_K          = 20    # number of nearest neighbours
KNN_BOTTOM_PCT = 5   # bottom percentile to mark on the histogram
KNN_SEED       = 0


def knn_density(pts, k):
    """
    kNN density estimate at each point.
      rho_i = (1 / mean_{j in kNN(i)} ||x_i - x_j||)^d
    pts : (n, d) array
    Returns (n,) array of density values (unnormalised).
    """
    d = pts.shape[1]
    tree = KDTree(pts)
    dists, _ = tree.query(pts, k=k + 1)   # k+1 to exclude the point itself
    mean_dist = dists[:, 1:].mean(axis=1)  # (n,)
    return (1.0 / mean_dist) ** d


# pick one random reconstruction from the chosen digit
rng_knn = np.random.RandomState(KNN_SEED)
labels_knn = labels.cpu().numpy() if isinstance(labels, torch.Tensor) else np.asarray(labels)
digit_idx = np.where(labels_knn == KNN_DIGIT)[0]
sample_idx = rng_knn.choice(digit_idx)

recon_pts = T_hat_all[sample_idx].cpu().numpy()  # (n, 2)
density   = knn_density(recon_pts, k=KNN_K)
thresh    = np.percentile(density, KNN_BOTTOM_PCT)

fig, (ax_hist, ax_scatter) = plt.subplots(1, 2, figsize=(12, 4))

ax_hist.hist(density, bins=40, color="steelblue", edgecolor="white", linewidth=0.4)
ax_hist.axvline(thresh, color="crimson", linestyle="--",
                label=f"Bottom {KNN_BOTTOM_PCT}%  ({thresh:.3g})")
ax_hist.set_xlabel("kNN density")
ax_hist.set_ylabel("Count")
ax_hist.set_title(f"Digit {KNN_DIGIT} reconstruction | k={KNN_K}")
ax_hist.legend()

lims = auto_lims(recon_pts)
sc = ax_scatter.scatter(recon_pts[:, 0], recon_pts[:, 1],
                        c=density, cmap="plasma", s=8, alpha=0.8)
set_lims(ax_scatter, lims)
ax_scatter.set_xticks([])
ax_scatter.set_yticks([])
ax_scatter.set_title(f"Density map (sample {sample_idx})")
plt.colorbar(sc, ax=ax_scatter, fraction=0.04, pad=0.02)

plt.tight_layout()
plt.show()


In [ ]:
# --- Reconstruction plot with kNN density filter ---
# Remove the bottom DENSITY_FILTER_PCT% of reconstructed points by kNN density,
# then plot original vs filtered reconstruction. Only Wasserstein is reported.

DENSITY_FILTER_PCT   = KNN_BOTTOM_PCT   # inherit from tuning cell, or override here
DENSITY_K            = KNN_K
DENSITY_RECON_DIGITS = None             # None = all digits
DENSITY_N_PER_CLASS  = 2
DENSITY_SEED         = 0
DENSITY_ALPHA        = 0.55


def w2_unequal(pts_a, pts_b):
    """W2 between two point clouds with (possibly different) uniform weights."""
    na, nb = len(pts_a), len(pts_b)
    a = np.full(na, 1.0 / na)
    b = np.full(nb, 1.0 / nb)
    cost = cdist(pts_a, pts_b, metric="sqeuclidean")
    return float(ot.emd2(a, b, cost)) ** 0.5


def density_filter(pts, k, bottom_pct):
    """Keep points above the bottom_pct percentile of kNN density."""
    rho = knn_density(pts, k=k)
    mask = rho >= np.percentile(rho, bottom_pct)
    return pts[mask]

def plot_reconstructions_density(
    maps, labels, recons, k, bottom_pct,
    n_per_class=1, alpha=0.6, digits=None, seed=0
):
    indices = pick_indices_per_class(
        labels,
        n_per_class=n_per_class,
        digits=digits,
        seed=seed
    )
    if not indices:
        raise ValueError("No reconstruction examples selected.")

    maps_np   = maps[indices].cpu().numpy()
    recons_np = recons[indices].cpu().numpy()
    labels_np = (
        labels[indices].cpu().numpy()
        if isinstance(labels, torch.Tensor)
        else np.asarray(labels)[indices]
    )

    fig, axes = plt.subplots(
        len(indices), 3,
        figsize=(15, 4 * len(indices)),
        squeeze=False
    )

    for row, (orig, recon, digit) in enumerate(zip(maps_np, recons_np, labels_np)):
        filtered = density_filter(recon, k=k, bottom_pct=bottom_pct)

        # keep same viewing window across all three panels
        lims = auto_lims(orig, recon)

        w2_raw = empirical_wasserstein_distance(orig, recon) ** 0.5
        w2_filtered = w2_unequal(orig, filtered)

        # original
        ax_l = axes[row, 0]
        ax_l.scatter(orig[:, 0], orig[:, 1], s=4, alpha=alpha, color="tab:blue")
        ax_l.set_title(f"Digit {digit} | original")
        set_lims(ax_l, lims)
        ax_l.set_xticks([])
        ax_l.set_yticks([])

        # raw reconstruction
        ax_m = axes[row, 1]
        ax_m.scatter(recon[:, 0], recon[:, 1], s=4, alpha=alpha, color="tab:orange")
        ax_m.set_title(f"Digit {digit} | reconstructed (W2={w2_raw:.4f})")
        set_lims(ax_m, lims)
        ax_m.set_xticks([])
        ax_m.set_yticks([])

        # filtered reconstruction
        ax_r = axes[row, 2]
        ax_r.scatter(filtered[:, 0], filtered[:, 1], s=4, alpha=alpha, color="tab:green")
        ax_r.set_title(
            f"Digit {digit} | reconstructed, filtered (W2={w2_filtered:.4f})"
        )
        set_lims(ax_r, lims)
        ax_r.set_xticks([])
        ax_r.set_yticks([])

    plt.tight_layout()
    plt.show()


plot_reconstructions_density(
    maps,
    labels,
    T_hat_all,
    k=DENSITY_K,
    bottom_pct=DENSITY_FILTER_PCT,
    n_per_class=DENSITY_N_PER_CLASS,
    alpha=DENSITY_ALPHA,
    digits=DENSITY_RECON_DIGITS,
    seed=DENSITY_SEED,
)


In [ ]:
# --- Average W2: baseline vs density-filtered reconstructions ---
# Note: computing emd2 for 5000 samples of size ~400 takes a few minutes.

maps_np_cmp   = maps.cpu().numpy()
recons_np_cmp = T_hat_all.cpu().numpy()
labels_np_cmp = labels.cpu().numpy() if isinstance(labels, torch.Tensor) else np.asarray(labels)

w2_base_all     = np.zeros(len(maps_np_cmp))
w2_filtered_all = np.zeros(len(maps_np_cmp))

for i in range(len(maps_np_cmp)):
    orig  = maps_np_cmp[i]
    recon = recons_np_cmp[i]
    w2_base_all[i]     = w2_unequal(orig, recon)
    filtered            = density_filter(recon, k=DENSITY_K, bottom_pct=DENSITY_FILTER_PCT)
    w2_filtered_all[i] = w2_unequal(orig, filtered)

print(f"Mean W2 — baseline:          {w2_base_all.mean():.6f}")
print(f"Mean W2 — density filtered:  {w2_filtered_all.mean():.6f}")
print()

col_w = 10
header = f"{'Digit':<7} {'Baseline':>{col_w}} {'Filtered':>{col_w}} {'Delta':>{col_w}}"
print(header)
print("─" * len(header))
for digit in sorted(np.unique(labels_np_cmp)):
    mask = labels_np_cmp == digit
    b = w2_base_all[mask].mean()
    f = w2_filtered_all[mask].mean()
    print(f"  {digit:<5} {b:{col_w}.6f} {f:{col_w}.6f} {f - b:+{col_w}.6f}")
